In [1]:
from pathlib import Path
from config import LOCAL_VOLUME, PROJECT_ROOT

import modal

app = modal.App("LLM-training")

container = (
    modal.Image.debian_slim(python_version="3.12")
    .uv_sync(uv_project_dir=PROJECT_ROOT)
    .add_local_python_source("transformer")
)

# same Volume name as etl.ipynb -- training reads the data/tokenizers ETL produced
volume = modal.Volume.from_name("LLM-pretraining", create_if_missing=True)

In [2]:
@app.cls(
    image=container,
    volumes={"/storage": volume},
    # cpu=1.0,
    gpu="T4",
    secrets=[modal.Secret.from_name("wandb-secret")],
)
class Trainer:
    """Training jobs for the pretraining pipeline. Reads data/tokenizers that
    etl.ipynb's ETL stages already committed to the Volume -- this class only
    trains, it doesn't prepare data.

    Layout (every path relative to VOLUME, same convention as ETL):
        data/{dataset_name}/bin/{tokenizer_uid}/{split}.bin   input  (from ETL)
        tokenizers/{tokenizer_uid}/...                        input  (from ETL)
        runs/{run_dir}/...                                    output (this class)

    `remote` drives everything location-specific: VOLUME (LOCAL_VOLUME vs the
    mount) and the commit/reload calls (Volume-API ops that only mean anything
    remotely -- locally we read/write disk directly).

    NOTE these are properties, not class attributes -- same reason as ETL: a
    class attribute is computed once at class-definition time (on the laptop,
    where is_local() is True), and that frozen value ships to the container
    as-is. A property re-evaluates is_local() at access time, on whichever
    side is actually running.
    """

    @property
    def remote(self) -> bool:
        return not modal.is_local()

    @property
    def VOLUME(self) -> Path:
        return Path("/storage") if self.remote else LOCAL_VOLUME

    @modal.method()
    def train(self, config_or_run_dir: dict | str, wandb_kwargs: dict | None = None):
        """Start a brand-new run (pass a config dict) or resume an existing one
        (pass its run_dir, relative to VOLUME) -- mirrors
        transformer.util.run_training's contract exactly, just dispatched
        through Modal instead of called directly.

        wandb_kwargs is passed straight through to run_training -- project,
        tags, entity, etc. are the caller's choice, not this class's; pass
        None to disable wandb entirely (e.g. for a local run with no
        wandb-secret env var to authenticate with)."""
        from transformer import run_training
        run_training(config_or_run_dir, self.VOLUME, wandb_kwargs=wandb_kwargs)


    @modal.method()
    def evaluate(self, run_dir: str):
        """Run eval/inference against a checkpoint under runs/{run_dir}/."""
        ...


In [ ]:
# --- Example driver: same local/remote toggle idiom as etl.ipynb ---
from transformer import TransformerLM
from transformer.optimizer import lr_cosine_schedule
import torch

dataset_name = 'gutenberg'
tokenizer_uid = 'gutenberg-bpe-1000'

config = {
    "description": "testrun2",
    "seed": 0,
    "model_class": TransformerLM,
    "model_params": {
        "vocab_size": 1000,
        "context_length": 16,
        "num_layers": 2,
        "d_model": 16,
        "d_ff": 32,
        "num_heads": 1,
        "rope_theta": 10000,
        "device": "cuda",
        "dtype": None,  # TODO: track this down - does this control all machine precision downstream?
    },
    "optimizer_class": torch.optim.AdamW,
    "optimizer_params": {
        "lr": 0.001,
        "betas": (0.9, 0.999),
        "weight_decay": 0.1,
        "eps": 1e-8,
    },
    "lr_schedule_fn": lr_cosine_schedule,
    "lr_schedule_params": {
        "max_learning_rate": 0.001,
        "min_learning_rate": 0.0001,
        "warmup_iters": 30,
        "cosine_cycle_iters": 500,
    },
    "training": {
        "train_path": f"data/{dataset_name}/bin/{tokenizer_uid}/train.bin",
        "valid_path": f"data/{dataset_name}/bin/{tokenizer_uid}/valid.bin",
        "batch_size": 64,
        "total_iterations": 1000,
        "val_every": 10,
        "save_every": 50,
    },
}

# passed straight through to run_training -- None disables wandb entirely
wandb_kwargs = {
    "project": "llm-pretraining",
    "tags": [dataset_name, tokenizer_uid],
}

# Flip this one variable: "remote" runs the job in a Modal container against
# the mounted Volume; "local" runs it in this process against LOCAL_VOLUME.
mode = "remote"  # "remote" | "local"


def run(method, *args):
    return getattr(method, mode)(*args)
config = 'runs/20260724T190204_testrun2_0'

with modal.enable_output():
    with app.run():
        trainer = Trainer()
        run(trainer.train, config, wandb_kwargs)